In [ ]:
#@title Install
import base64

encoded_text = "Q3JlYXRlIGJ5IDogYWlnb2xkZW4="
decoded_text = base64.b64decode(encoded_text.encode()).decode()

print(decoded_text)

print("📦 Installing dependencies...")
!pip install google-genai pydub
!apt-get install -y ffmpeg

print("✅ Install successfully!")



In [ ]:
# @title Colab-specific setup:

import base64
import mimetypes
import os
import re
import struct
import time
import zipfile
from google import genai
from google.genai import types
from IPython.display import Audio, display
from google.colab import userdata
from google.colab import files
try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("⚠️ pydub در دسترس نیست. فایل‌های صوتی به صورت جداگانه ذخیره می‌شوند.")

gemini_api_key_input = "" #@param {type:"string", input_type:"password"}

# @markdown ---
# @markdown ### انتخاب روش ورودی
use_file_input = False #@param {type:"boolean"}
# @markdown اگر تیک بالا را فعال کنید، فایل متنی آپلود شده فقط شامل متن اصلی خواهد بود (پرامپت از فیلد زیر خوانده می‌شود)

# @markdown ---
# @markdown ### تنظیمات پرامپت (همیشه از این فیلد استفاده می‌شود)
# @markdown پرامپت برای تنظیم سبک گفتار (مثال: "از زبان یک یوتوبر پر انرژی و حرفه ای"):
speech_prompt = "" #@param {type:"string"}

# @markdown ---
# @markdown ### متن ورودی (فقط در صورت غیرفعال بودن گزینه فایل)
# @markdown متن مورد نظر برای تبدیل به گفتار:
text_to_speak = "" #@param {type:"string"}

# @markdown ---
# @markdown ### تنظیمات تقسیم‌بندی متن
# @markdown حداکثر تعداد کاراکتر در هر قطعه:
max_chunk_size = 3800 #@param {type:"slider", min:2000, max:4000, step:100}
# @markdown فاصله زمانی بین درخواست‌ها (ثانیه):
sleep_between_requests = 12 #@param {type:"slider", min:10, max:15, step:0.5}

# @markdown ---
# @markdown ### تنظیمات مدل
# @markdown دمای مدل (تاثیر بر تنوع خروجی):
temperature = 1 #@param {type:"slider", min:0, max:2, step:0.05}

# @markdown ---
# @markdown ### انتخاب مدل و گوینده و فایل خروجی
# @markdown مدل مورد نظر را انتخاب کنید:
model_name = "gemini-3.1-flash-tts-preview" #@param ["gemini-3.1-flash-tts-preview", "gemini-2.5-flash-preview-tts", "gemini-2.5-pro-preview-tts"]
# @markdown گوینده مورد نظر را از لیست انتخاب کنید:
speaker_voice = "Charon" #@param ["Achird", "Zubenelgenubi", "Vindemiatrix", "Sadachbia", "Sadaltager", "Sulafat", "Laomedeia", "Achernar", "Alnilam", "Schedar", "Gacrux", "Pulcherrima", "Umbriel", "Algieba", "Despina", "Erinome", "Algenib", "Rasalthgeti", "Orus", "Aoede", "Callirrhoe", "Autonoe", "Enceladus", "Iapetus", "Zephyr", "Puck", "Charon", "Kore", "Fenrir", "Leda"]

# @markdown نام فایل خروجی (بدون پسوند):
output_filename_base = "gemini_tts_output" #@param {type:"string"}

# @markdown ---
# @markdown ### تنظیمات خروجی

merge_audio_files = True #@param {type:"boolean"}
# @markdown حذف فایل‌های جزئی پس از ادغام:
delete_partial_files = False #@param {type:"boolean"}

# --- Helper functions ---

def save_binary_file(file_name, data):
    with open(file_name, "wb") as f:
        f.write(data)
    print(f"✅ فایل در مسیر زیر ذخیره شد: {file_name}")
    return file_name

def convert_to_wav(audio_data: bytes, mime_type: str) -> bytes:
    parameters = parse_audio_mime_type(mime_type)
    bits_per_sample = parameters["bits_per_sample"]
    sample_rate = parameters["rate"]
    num_channels = 1
    data_size = len(audio_data)
    bytes_per_sample = bits_per_sample // 8
    block_align = num_channels * bytes_per_sample
    byte_rate = sample_rate * block_align
    chunk_size = 36 + data_size

    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF",
        chunk_size,
        b"WAVE",
        b"fmt ",
        16,
        1,
        num_channels,
        sample_rate,
        byte_rate,
        block_align,
        bits_per_sample,
        b"data",
        data_size
    )
    return header + audio_data

def parse_audio_mime_type(mime_type: str) -> dict[str, int | None]:
    bits_per_sample = 16
    rate = 24000

    parts = mime_type.split(";")
    for param in parts:
        param = param.strip()
        if param.lower().startswith("rate="):
            try:
                rate_str = param.split("=", 1)[1]
                rate = int(rate_str)
            except (ValueError, IndexError):
                pass
        elif param.startswith("audio/L"):
            try:
                bits_per_sample = int(param.split("L", 1)[1])
            except (ValueError, IndexError):
                pass
    return {"bits_per_sample": bits_per_sample, "rate": rate}

def load_text_file():
    """Load text file containing only the main text (no prompt)"""
    print("📁 لطفاً فایل متنی خود را آپلود کنید...")
    print("💡 فایل فقط باید شامل متن اصلی باشد (پرامپت از فیلد بالا خوانده می‌شود)")

    uploaded = files.upload()

    if not uploaded:
        print("❌ هیچ فایلی آپلود نشد.")
        return ""

    file_name = list(uploaded.keys())[0]
    print(f"✅ فایل '{file_name}' با موفقیت آپلود شد.")

    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            content = f.read().strip()

        print(f"📖 متن بارگذاری شده: {len(content)} کاراکتر")
        print(f"📝 نمونه متن: '{content[:100]}{'...' if len(content) > 100 else ''}'")

        return content

    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return ""

def smart_text_split(text, max_size=3800):
    """Split text into chunks without breaking sentences"""
    if len(text) <= max_size:
        return [text]

    chunks = []
    current_chunk = ""

    sentences = re.split(r'(?<=[.!?])\s+', text)

    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_size:
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence
            else:
                words = sentence.split()
                temp_chunk = ""
                for word in words:
                    if len(temp_chunk) + len(word) + 1 > max_size:
                        if temp_chunk:
                            chunks.append(temp_chunk.strip())
                            temp_chunk = word
                        else:
                            chunks.append(word[:max_size])
                            word = word[max_size:]
                            while len(word) > max_size:
                                chunks.append(word[:max_size])
                                word = word[max_size:]
                            if word:
                                temp_chunk = word
                    else:
                        temp_chunk += (" " if temp_chunk else "") + word
                current_chunk = temp_chunk
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

def merge_audio_files_func(file_paths, output_path):
    """Merge multiple audio files into one"""
    if not PYDUB_AVAILABLE:
        print("❌ pydub در دسترس نیست. نمی‌توان فایل‌ها را ادغام کرد.")
        return False

    try:
        print(f"🔗 در حال ادغام {len(file_paths)} فایل صوتی...")

        combined = AudioSegment.empty()

        for i, file_path in enumerate(file_paths):
            if os.path.exists(file_path):
                print(f"📎 اضافه کردن فایل {i+1}: {file_path}")
                audio = AudioSegment.from_file(file_path)
                combined += audio
                if i < len(file_paths) - 1:
                    combined += AudioSegment.silent(duration=500)
            else:
                print(f"⚠️ فایل پیدا نشد: {file_path}")

        combined.export(output_path, format="wav")
        print(f"✅ فایل ادغام شده ذخیره شد: {output_path}")
        return True

    except Exception as e:
        print(f"❌ خطا در ادغام فایل‌ها: {e}")
        return False

def create_zip_file(file_paths, zip_name):
    """Create a zip file containing all audio files"""
    try:
        with zipfile.ZipFile(zip_name, 'w') as zipf:
            for file_path in file_paths:
                if os.path.exists(file_path):
                    zipf.write(file_path, os.path.basename(file_path))
        print(f"📦 فایل ZIP ایجاد شد: {zip_name}")
        return True
    except Exception as e:
        print(f"❌ خطا در ایجاد فایل ZIP: {e}")
        return False

# --- Main generation function ---

def generate_audio_from_text(text_input, prompt_input, selected_voice, output_base_name,
                           api_key_input_field, model, temperature, use_file=False,
                           max_chunk=3800, sleep_time=2, merge_files=True, delete_partials=True):
    print("🚀 شروع فرآیند تبدیل متن به گفتار...")

    # Handle file input if enabled
    if use_file:
        print("📁 حالت فایل فعال است. در حال آپلود فایل...")
        file_text = load_text_file()
        if not file_text:
            print("❌ خطا: متن استخراج شده از فایل خالی است.")
            return
        text_input = file_text
        print("✅ متن از فایل با موفقیت بارگذاری شد.")
    else:
        print("⌨️ حالت ورودی دستی فعال است.")

    # 1. API Key Retrieval and Validation
    api_key = None
    if api_key_input_field:
        api_key = api_key_input_field
        print("🔑 کلید API از فیلد ورودی بارگذاری شد.")
    else:
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key:
            print("🔑 کلید API از Colab Secrets بارگذاری شد.")
        else:
            print("❌ خطا: کلید API جمینای (GEMINI_API_KEY) پیدا نشد.")
            print("لطفاً کلید API خود را در فیلد بالا وارد کنید یا مطمئن شوید که آن را در Colab Secrets به درستی ذخیره کرده‌اید.")
            print("💡 راهنما: روی آیکون قفل 🔑 در نوار کناری سمت چپ Colab کلیک کنید، یک 'راز' (Secret) جدید با نام `GEMINI_API_KEY` ایجاد کنید و کلید خود را به عنوان مقدار آن قرار دهید. سپس 'Notebook access' را فعال کنید.")
            return

    os.environ["GEMINI_API_KEY"] = api_key
    print("🔧 متغیر محیطی GEMINI_API_KEY تنظیم شد.")

    # 2. Initialize GenAI Client
    try:
        print("🛠️ در حال ایجاد کلاینت جمینای...")
        client = genai.Client(
            api_key=os.environ.get("GEMINI_API_KEY"),
        )
        print("✅ کلاینت جمینای با موفقیت ایجاد شد.")
    except Exception as e:
        print(f"❌ خطا در ایجاد کلاینت جمینای: {e}")
        print("لطفاً از صحت کلید API خود اطمینان حاصل کنید. ممکن است کلید منقضی شده یا نامعتبر باشد.")
        return

    # 3. Validate Text Input
    if not text_input or text_input.strip() == "":
        print("❌ خطا: متن ورودی برای تبدیل به گفتار خالی است. لطفاً متنی را وارد کنید.")
        return

    # 4. Split text into chunks
    text_chunks = smart_text_split(text_input, max_chunk)
    print(f"📊 متن به {len(text_chunks)} قطعه تقسیم شد.")

    for i, chunk in enumerate(text_chunks):
        print(f"📝 قطعه {i+1}: {len(chunk)} کاراکتر")

    # 5. Generate audio for each chunk
    generated_files = []

    for i, chunk in enumerate(text_chunks):
        print(f"\n🔊 تولید صدا برای قطعه {i+1}/{len(text_chunks)}...")

        final_text = f'"{prompt_input}"\n{chunk}' if prompt_input.strip() else chunk

        contents = [
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=final_text),
                ],
            ),
        ]
        generate_content_config = types.GenerateContentConfig(
            temperature=temperature,
            response_modalities=[
                "audio",
            ],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name=selected_voice
                    )
                )
            ),
        )

        try:
            chunk_filename = f"{output_base_name}_part_{i+1:03d}"

            for chunk_data in client.models.generate_content_stream(
                model=model,
                contents=contents,
                config=generate_content_config,
            ):
                if (
                    chunk_data.candidates
                    and chunk_data.candidates[0].content
                    and chunk_data.candidates[0].content.parts
                    and chunk_data.candidates[0].content.parts[0].inline_data
                ):
                    inline_data = chunk_data.candidates[0].content.parts[0].inline_data
                    data_buffer = inline_data.data
                    file_extension = mimetypes.guess_extension(inline_data.mime_type)
                    if file_extension is None:
                        file_extension = ".wav"
                        data_buffer = convert_to_wav(inline_data.data, inline_data.mime_type)

                    generated_file_path = save_binary_file(f"{chunk_filename}{file_extension}", data_buffer)
                    generated_files.append(generated_file_path)
                    print(f"✅ قطعه {i+1} تولید شد: {generated_file_path}")
                    break
                else:
                    if chunk_data.text:
                        print(f"ℹ️ پیام متنی از API: {chunk_data.text}")

        except Exception as e:
            print(f"❌ خطا در تولید قطعه {i+1}: {e}")
            continue

        if i < len(text_chunks) - 1:
            print(f"⏱️ انتظار {sleep_time} ثانیه...")
            time.sleep(sleep_time)

    # 6. Handle output files
    if not generated_files:
        print("❌ هیچ فایل صوتی تولید نشد!")
        return

    print(f"\n🎉 {len(generated_files)} فایل صوتی با موفقیت تولید شد!")

    final_audio_file = None
    if merge_files and len(generated_files) > 1:
        merged_filename = f"{output_base_name}_merged.wav"
        if merge_audio_files_func(generated_files, merged_filename):
            final_audio_file = merged_filename
            print(f"🎵 فایل نهایی ادغام شده: {merged_filename}")

            if delete_partials:
                for file_path in generated_files:
                    try:
                        os.remove(file_path)
                        print(f"🗑️ فایل جزئی حذف شد: {file_path}")
                    except:
                        pass
        else:
            print("⚠️ ادغام ممکن نبود. فایل‌های جداگانه حفظ شدند.")

    if not final_audio_file and len(generated_files) > 1:
        zip_filename = f"{output_base_name}_all_parts.zip"
        create_zip_file(generated_files, zip_filename)

    # 7. Play the audio
    if final_audio_file and os.path.exists(final_audio_file):
        print(f"▶️ پخش فایل ادغام شده: {final_audio_file}")
        display(Audio(final_audio_file, autoplay=True))
    elif generated_files and os.path.exists(generated_files[0]):
        print(f"▶️ پخش اولین قطعه: {generated_files[0]}")
        display(Audio(generated_files[0], autoplay=True))
    else:
        print("🛑 پخش صدا امکان‌پذیر نیست زیرا فایل صوتی تولید نشده است.")

# --- Run the generation ---
if __name__ == "__main__":
    if use_file_input:
        print("📁 حالت فعال: آپلود فایل متنی")
        print("ℹ️ پرامپت از فیلد بالا خوانده می‌شود.")
    else:
        print("⌨️ حالت فعال: ورودی دستی")
        if not text_to_speak.strip():
            print("⚠️ هشدار: متن ورودی خالی است. لطفاً متن را وارد کنید یا حالت فایل را فعال کنید.")

    generate_audio_from_text(
        text_to_speak,
        speech_prompt,
        speaker_voice,
        output_filename_base,
        gemini_api_key_input,
        model_name,
        temperature,
        use_file_input,
        max_chunk=max_chunk_size,
        sleep_time=sleep_between_requests,
        merge_files=merge_audio_files,
        delete_partials=delete_partial_files
    )

In [ ]:
#@title Download Final Audio File
from google.colab import drive
import os
import glob
import shutil

# اتصال به Google Drive
drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/gemini"

os.makedirs(drive_folder, exist_ok=True)

matching_files = glob.glob("/content/gemini*")

if matching_files:
    for file_path in matching_files:
        shutil.copy(file_path, drive_folder)
        print(f"✅ کپی شد: {os.path.basename(file_path)}")
    print(f"\n📁 همه فایل‌ها داخل: {drive_folder}")
else:
    print("❌ هیچ فایلی با شروع gemini پیدا نشد.")

In [ ]:
#@title Delete
import os
import shutil

colab_path = "/content"
excluded_folders = ["sample_data", "drive"]

for item in os.listdir(colab_path):
    item_path = os.path.join(colab_path, item)

    if item not in excluded_folders:
        try:
            if os.path.isfile(item_path):
                os.remove(item_path)
                print(f"Deleted file: {item_path}")
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"Deleted folder: {item_path}")
        except Exception as e:
            print(f"Error deleting {item_path}: {e}")

print("تمام فایل‌ها حذف شدند.")
